In [0]:
from pyspark.sql import functions as F

In [0]:
df_bronze = spark.table("workspace.default.capstone_bronze_sales")
# dataCount = df_bronze.count()

In [0]:
noDup_df = df_bronze.dropDuplicates(["order_id"])

string_cols = [f.name for f in df_bronze.schema.fields if f.dataType.typeName() == "string"]
trim_expressions = {
    col_name: F.when(F.trim(F.col(col_name)) == "", F.lit(None))
               .otherwise(F.trim(F.col(col_name)))
    for col_name in string_cols
}
df_trim = df_bronze.withColumns(trim_expressions)

In [0]:
df_quality = df_trim.withColumn(
    "quality_flag",
    F.when(F.col("customer_id").isNull(), "INVALID_CUSTOMER")
     .when(F.col("quantity") <= 0, "INVALID_QUANTITY")
     .when(F.col("net_amount") < 0, "INVALID_AMOUNT")
     .when(F.upper(F.trim(F.col("product_id"))) == "UNKNOWN", "INVALID_PRODUCT")
     .otherwise("VALID")
)

df_valid = df_quality.filter(F.col("quality_flag") == "VALID")
df_invalid = df_quality.filter(F.col("quality_flag") != "VALID")

In [0]:
df_silver = df_valid \
    .withColumn("order_id", F.col("order_id").cast("string")) \
    .withColumn("customer_id", F.col("customer_id").cast("string")) \
    .withColumn("customer_name", F.col("customer_name").cast("string")) \
    .withColumn("city", F.col("city").cast("string")) \
    .withColumn("state", F.col("state").cast("string")) \
    .withColumn("product_id", F.col("product_id").cast("string")) \
    .withColumn("product_name", F.col("product_name").cast("string")) \
    .withColumn("category", F.col("category").cast("string")) \
    .withColumn("payment_method", F.col("payment_method").cast("string")) \
    .withColumn("order_status", F.col("order_status").cast("string")) \
    .withColumn("quantity", F.col("quantity").cast("int")) \
    .withColumn("unit_price", F.col("unit_price").cast("double")) \
    .withColumn("discount_pct", F.col("discount_pct").cast("double")) \
    .withColumn("gross_amount", F.col("gross_amount").cast("double")) \
    .withColumn("discount_amount", F.col("discount_amount").cast("double")) \
    .withColumn("net_amount", F.col("net_amount").cast("double")) \
    .withColumn("order_date", F.to_date(F.col("order_date"), "yyyy-MM-dd")) \
    .withColumn("year", F.year(F.col("order_date"))) \
    .withColumn("month", F.month(F.col("order_date"))) \
    .withColumn("month_name", F.date_format(F.col("order_date"), "MMMM"))

# df_silver.show()

df_silver.write.mode("overwrite").saveAsTable("workspace.default.capstone_silver_sales")